In [0]:
bucket = "phase8-databricks-lakehouse"

raw_orders_path = f"s3://{bucket}/raw/orders/"
bronze_path = f"s3://{bucket}/curated/bronze/orders/"
silver_path = f"s3://{bucket}/curated/silver/orders/"
gold_path = f"s3://{bucket}/curated/gold/daily_sales/"
bronze_checkpoint = f"s3://{bucket}/checkpoints/bronze_orders/"
silver_checkpoint = f"s3://{bucket}/checkpoints/silver_orders/"

spark.sql("CREATE SCHEMA IF NOT EXISTS phase8_lab")
spark.sql("USE phase8_lab")

DataFrame[]

In [0]:
from pyspark.sql.functions import current_timestamp, col

bronze_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", bronze_checkpoint + "_schema")
         .option("header", "true")
         .load(raw_orders_path)
         .withColumn("source_file", col("_metadata.file_path"))
         .withColumn("ingestion_ts", current_timestamp())
)

In [0]:
bronze_query = (
    bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", bronze_checkpoint)
        .trigger(availableNow=True)
        .toTable("bronze_orders")
)

In [0]:
display(spark.sql("""
SELECT order_id, customer_id, order_ts, store_region, status, source_file, ingestion_ts
FROM bronze_orders
ORDER BY order_ts
LIMIT 20
"""))

order_id,customer_id,order_ts,store_region,status,source_file,ingestion_ts
CO000001,CC02163,2025-01-01T00:00:00,North,Cancelled,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000002,CC02643,2025-01-01T00:01:00,South,Placed,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000003,CC03423,2025-01-01T00:02:00,South,Placed,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000004,CC01205,2025-01-01T00:03:00,South,Placed,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000005,CC04551,2025-01-01T00:04:00,South,Delivered,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000005,CC04551,2025-01-01T00:04:00,South,Delivered,s3://phase8-databricks-lakehouse/raw/orders/orders_day2.json,2026-07-19T15:34:52.128Z
CO000006,CC02727,2025-01-01T00:05:00,North,Delivered,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000006,CC02727,2025-01-01T00:05:00,North,Delivered,s3://phase8-databricks-lakehouse/raw/orders/orders_day2.json,2026-07-19T15:34:52.128Z
CO000007,CC02108,2025-01-01T00:06:00,North,Cancelled,s3://phase8-databricks-lakehouse/raw/orders/orders_day1.json,2026-07-19T15:34:52.128Z
CO000007,CC02108,2025-01-01T00:06:00,North,Cancelled,s3://phase8-databricks-lakehouse/raw/orders/orders_day2.json,2026-07-19T15:34:52.128Z


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_orders (
  order_id STRING,
  customer_id STRING,
  order_ts TIMESTAMP,
  store_region STRING,
  status STRING,
  source_file STRING,
  ingestion_ts TIMESTAMP
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import upper, trim, to_timestamp, row_number
from pyspark.sql.window import Window

valid_statuses = ["Placed", "Shipped", "Delivered", "Cancelled"]

silver_df = (
    spark.table("bronze_orders")
         .withColumn("order_status", upper(trim(col("status"))))
         .withColumn("order_ts", to_timestamp(col("order_ts")))
         .filter(col("order_id").isNotNull())
         .filter(col("customer_id").isNotNull())
         .filter(col("order_status").isin(valid_statuses))
)

dedupe_window = Window.partitionBy("order_id").orderBy(col("order_ts").desc(), col("ingestion_ts").desc())

silver_deduped = (
    silver_df.withColumn("rn", row_number().over(dedupe_window))
             .filter(col("rn") == 1)
             .drop("rn")
)

In [0]:
silver_deduped.createOrReplaceTempView("silver_updates")

spark.sql("""
MERGE INTO silver_orders AS target
USING silver_updates AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS invalid_rows
FROM silver_orders
WHERE order_id IS NULL
   OR customer_id IS NULL
   OR status NOT IN ('Placed','Shipped','Delivered','Cancelled')
"""))

display(spark.sql("""
SELECT order_id, COUNT(*) AS row_count
FROM silver_orders
GROUP BY order_id
HAVING COUNT(*) > 1
"""))

display(spark.sql("DESCRIBE DETAIL silver_orders"))

invalid_rows
0


order_id,row_count


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,19fb9cad-7f69-432b-8579-45c9f472416b,dbx_phase8.phase8_lab.silver_orders,null,s3://databricks-storage-7474660680123943/unity-catalog/7474660680123943/__unitystorage/catalogs/98a31a9e-f7fa-4d17-814d-a777dd0af618/tables/b2db22df-c469-4022-9c30-8a341ef27f38,2026-07-19T15:35:46.533Z,2026-07-19T15:43:48Z,List(),List(),1,312435,"Map(delta.enableChangeDataFeed -> true, delta.enableDeletionVectors -> true)",3,7,"List(changeDataFeed, deletionVectors)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


In [0]:
from pyspark.sql.functions import to_date, sum as _sum, count, countDistinct

gold_df = (
    spark.table("silver_orders")
         .withColumn("order_date", to_date(col("order_ts")))
         .groupBy("order_date", "status")
         .agg(
             count("order_id").alias("total_orders"),
             countDistinct("customer_id").alias("distinct_customers")
         )
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_daily_sales")

In [0]:
display(spark.sql("""
SELECT *
FROM gold_daily_sales
ORDER BY order_date, status
LIMIT 50
"""))

order_date,status,total_orders,distinct_customers
2025-01-01,Cancelled,371,354
2025-01-01,Delivered,370,352
2025-01-01,Placed,366,353
2025-01-01,Shipped,333,323
2025-01-02,Cancelled,368,359
2025-01-02,Delivered,328,320
2025-01-02,Placed,381,363
2025-01-02,Shipped,363,348
2025-01-03,Cancelled,352,338
2025-01-03,Delivered,370,355


In [0]:
display(spark.sql("DESCRIBE HISTORY silver_orders"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-19T15:43:48Z,70635340563176,kukatlasusindhar27@gmail.com,MERGE,"Map(predicate -> [""(order_id#4630 = order_id#4429)""], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2690817680640659),0719-151555-kimmphqo,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 312435, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4117, materializeSourceTimeMs -> 1706, numTargetRowsInserted -> 20000, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1125, numTargetRowsUpdated -> 0, numOutputRows -> 20000, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 20000, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1209)",null,Databricks-Runtime/14.3.x-scala2.12
1,2026-07-19T15:37:01Z,70635340563176,kukatlasusindhar27@gmail.com,MERGE,"Map(predicate -> [""(order_id#1907 = order_id#1714)""], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2690817680640659),0719-151555-kimmphqo,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 8736, materializeSourceTimeMs -> 2343, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4131, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1971)",null,Databricks-Runtime/14.3.x-scala2.12
0,2026-07-19T15:35:48Z,70635340563176,kukatlasusindhar27@gmail.com,CREATE TABLE,"Map(partitionBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableChangeDataFeed"":""true"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2690817680640659),0719-151555-kimmphqo,null,WriteSerializable,true,Map(),null,Databricks-Runtime/14.3.x-scala2.12


In [0]:
display(spark.sql("""
SELECT *
FROM silver_orders VERSION AS OF 0
ORDER BY order_ts
"""))

order_id,customer_id,order_ts,store_region,status,source_file,ingestion_ts


In [0]:
cdf_df = (
    spark.read.format("delta")
         .option("readChangeFeed", "true")
         .option("startingVersion", 0)
         .table("silver_orders")
)

display(cdf_df.select("order_id", "status", "_change_type", "_commit_version").limit(50))

order_id,status,_change_type,_commit_version
CO000001,Cancelled,insert,2
CO000002,Placed,insert,2
CO000003,Placed,insert,2
CO000004,Placed,insert,2
CO000005,Delivered,insert,2
CO000006,Delivered,insert,2
CO000007,Cancelled,insert,2
CO000008,Shipped,insert,2
CO000009,Cancelled,insert,2
CO000010,Placed,insert,2


In [0]:
changed_dates = [r["order_date"] for r in (
    cdf_df.withColumn("order_date", to_date(col("order_ts")))
          .select("order_date")
          .distinct()
          .collect()
) if r["order_date"] is not None]

incremental_gold = (
    spark.table("silver_orders")
         .withColumn("order_date", to_date(col("order_ts")))
         .filter(col("order_date").isin(changed_dates))
         .groupBy("order_date", "status")
         .agg(
             count("order_id").alias("total_orders"),
             countDistinct("customer_id").alias("distinct_customers")
         )
)

incremental_gold.createOrReplaceTempView("gold_updates")

spark.sql("""
MERGE INTO gold_daily_sales AS target
USING gold_updates AS source
ON target.order_date = source.order_date AND target.status = source.status
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
print(f"Current catalog: {current_catalog}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {current_catalog}.governance_lab")

Current catalog: dbx_phase8


DataFrame[]

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {current_catalog}.governance_lab.silver_orders_uc
AS SELECT * FROM phase8_lab.silver_orders
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {current_catalog}.governance_lab.gold_sales_metrics_uc
AS SELECT * FROM phase8_lab.gold_daily_sales
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Capture your current login identity automatically
current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Assigning persona grants to current user principal: {current_user}")

# Granting Data Engineer style access to yourself
spark.sql(f"GRANT USE CATALOG ON CATALOG {current_catalog} TO `{current_user}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {current_catalog}.governance_lab TO `{current_user}`")
spark.sql(f"GRANT SELECT ON TABLE {current_catalog}.governance_lab.silver_orders_uc TO `{current_user}`")
spark.sql(f"GRANT SELECT ON TABLE {current_catalog}.governance_lab.gold_sales_metrics_uc TO `{current_user}`")

Assigning persona grants to current user principal: kukatlasusindhar27@gmail.com


DataFrame[]

In [0]:
spark.sql("CREATE SHARE IF NOT EXISTS governance_gold_share")

DataFrame[info_name: string, info_value: string]

In [0]:
try:
    spark.sql(f"""
    ALTER SHARE governance_gold_share
    ADD TABLE {current_catalog}.governance_lab.gold_sales_metrics_uc
    WITH HISTORY
    """)
    print("Table successfully added to the share!")
except Exception as e:
    if "ALREADY_EXISTS" in str(e):
        print("Table is already in the share. Safe to continue!")
    else:
        raise e

Table successfully added to the share!


In [0]:
spark.sql("""
CREATE RECIPIENT IF NOT EXISTS governance_external_recipient
""")

DataFrame[info_name: string, info_value: string]

In [0]:
display(spark.sql("SHOW SHARES"))
display(spark.sql("DESCRIBE SHARE governance_gold_share"))

share,created_at,created_by,comment
governance_gold_share,2026-07-18T16:42:22.187Z,kukatlasusindhar27@gmail.com,null


info_name,info_value
share_name,governance_gold_share
owner,kukatlasusindhar27@gmail.com
created_at,2026-07-18 16:42:22.187
created_by,kukatlasusindhar27@gmail.com
updated_at,2026-07-19 17:01:55.086
updated_by,kukatlasusindhar27@gmail.com
comment,
